In [1]:
!pip install torchxrayvision -q

import torchxrayvision as xrv
import torch

# Load pretrained CXR8 model
model = xrv.models.DenseNet(weights="densenet121-res224-nih")
model.eval()

print("Model loaded")
print(f"Pathologies ({len(model.pathologies)}):")
for i, p in enumerate(model.pathologies):
    print(f"   {i+1:>2}. {p}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 65.1 MB/s eta 0:00:00:00:0100:01
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
Model loaded
Pathologies (18):
    1. Atelectasis
    2. Consolidation
    3. Infiltration
    4. Pneumothorax
    5. Edema
    6. Emphysema
    7. Fibrosis
    8. Effusion
    9. Pneumonia
   10. Pleural_Thickening
   11. Cardiomegaly
   12. Nodule
   13. Mass
   14. Hernia
   15. 
   16. 
   17. 
   18. 


In [2]:
import torchxrayvision as xrv
import torchvision.transforms as transforms
import torch
import numpy as np
import skimage.io

# test image from CXR8
import glob
DATA_DIR = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
sample_img_path = glob.glob(f"{DATA_DIR}/images_001/images/*.png")[0]
print(f"Testing on: {sample_img_path}")

# Load & preprocess 
img = skimage.io.imread(sample_img_path)
img = xrv.datasets.normalize(img, 255)

# Handle grayscale vs RGB
if len(img.shape) == 3:
    img = img.mean(axis=2)  # convert to grayscale

img = img[None, ...]  # add channel dim → (1, H, W)

transform = transforms.Compose([
    xrv.datasets.XRayCenterCrop(),
    xrv.datasets.XRayResizer(224)
])
img = transform(img)
img_tensor = torch.from_numpy(img).unsqueeze(0)  # (1, 1, 224, 224)

print(f"Image shape : {img_tensor.shape}")

# Inference 
with torch.no_grad():
    preds = model(img_tensor).squeeze()

Testing on: /kaggle/input/datasets/organizations/nih-chest-xrays/data/images_001/images/00000502_003.png
Image shape : torch.Size([1, 1, 224, 224])


In [4]:
import pandas as pd

# Check what the actual label is for this image
DATA_DIR = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
df = pd.read_csv(f"{DATA_DIR}/Data_Entry_2017.csv")

img_name = "00000502_003.png"
row = df[df["Image Index"] == img_name]
print(f"Image : {img_name}")
print(f"True label : {row['Finding Labels'].values[0]}")

Image : 00000502_003.png
True label : Infiltration
